# SafeQuest — Guardian Presentation-Style Classifier
### Training, cleaning and evaluation notebook (Google Colab)

Run the cells top to bottom. Every cell prints a result or draws a chart you can
screenshot for your report.

**Runtime → Change runtime type → CPU is fine.** This does not need a GPU.

## 1 · Setup

In [ ]:
!pip install -q scikit-image skl2onnx onnx 2>/dev/null
print("packages ready")

In [ ]:
# Download the pipeline module (or upload guardian_pipeline.py yourself)
from google.colab import files
import os
if not os.path.exists("guardian_pipeline.py"):
    print("Upload guardian_pipeline.py now:")
    files.upload()
import guardian_pipeline as gp
print("module loaded")

## 2 · Load your dataset

**Option A — upload a zip** (best for ~1000 images)
**Option B — mount Google Drive** if the data is already there.

Run *one* of the two cells below.

In [ ]:
# OPTION A: upload a zip of your image folder
from google.colab import files
import zipfile, os
up = files.upload()                      # pick your .zip
name = list(up.keys())[0]
os.makedirs("data", exist_ok=True)
with zipfile.ZipFile(name) as z:
    z.extractall("data")
DATA_DIR = "data"
print("extracted to ./data")

In [ ]:
# OPTION B: use Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/your_dataset_folder'

## 3 · Load and audit

The loader auto-detects your labelling scheme — UTKFace filenames
(`25_0_1_xxx.jpg`) or folders named `male/` and `female/`. It also walks past the
nested `dataset/dataset/` folders that Kaggle zips always create.

In [ ]:
from collections import Counter
import numpy as np, json

rows, audit = gp.load_dataset(DATA_DIR)
balance_before = dict(Counter(r["label"] for r in rows))

print("Layout detected :", audit["layout"])
print("Image files found:", audit["files_found"])
print("Labelled         :", audit["labelled"])
print("Unlabelled skipped:", audit["unlabelled_skipped"])
print("Class balance    : masculine=%d  feminine=%d"
      % (balance_before.get(0,0), balance_before.get(1,0)))

## 4 · Data cleaning

Five rejection rules, each counted so you can show exactly what was removed:

| rule | removes |
|---|---|
| `unreadable` | corrupt / truncated files |
| `too_small` | images under 32 px |
| `too_blurry` | Laplacian variance below threshold |
| `duplicate` | 16×16 difference-hash collisions |
| `no_face_detected` | Haar cascade found no face |

In [ ]:
X, y, groups, kept, reasons = gp.clean_and_featurise(
    rows,
    detect=True,        # set False if your images are already tight face crops
    min_size=32,
    blur_thresh=12.0,
)
balance_after = dict(Counter(y.tolist()))

print("\nCLEANING SUMMARY")
for k, v in reasons.items():
    print(f"  {k:<20} {v}")
print(f"\nFeature matrix: {X.shape[0]} images x {X.shape[1]} HOG features")
print("Class balance after: masculine=%d  feminine=%d"
      % (balance_after.get(0,0), balance_after.get(1,0)))

### Chart 1 — class balance before vs after cleaning

In [ ]:
gp.plot_class_balance(balance_before, balance_after);

### Chart 2 — cleaning funnel

In [ ]:
gp.plot_cleaning_funnel(audit, reasons);

### Chart 3 — preprocessing, raw vs cleaned crop

In [ ]:
gp.plot_samples(kept, n=8);

### Chart 4 — HOG features

This shows what the model actually looks at: gradient orientations, not colour.
Good demo slide — it makes the model non-magical.

In [ ]:
gp.plot_hog(kept);

## 5 · BEFORE training — baseline scores

This is your "before" evidence. Three models that have learned nothing:
random guessing, always predicting the majority class, and a logistic
regression with random weights. Anything you build has to beat these.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(
    X, y, groups, test_size=0.2, stratify=y, random_state=gp.SEED)
print(f"train = {len(X_tr)}   test = {len(X_te)}")

baselines = gp.run_baselines(X_tr, y_tr, X_te, y_te)
print("\nBASELINE (untrained) SCORES")
for k, v in baselines.items():
    print(f"  {k.replace(chr(10),' '):<20} acc={v['accuracy']:.4f}  macroF1={v['macro_f1']:.4f}")

## 6 · Tune regularisation

With ~1000 images and 1764 features the default `C=1.0` overfits hard. This
sweep typically buys 5–7 points of macro F1 — worth its own paragraph in the
report.

In [ ]:
best_C, cv_means, cv_stds = gp.tune_C(X_tr, y_tr)

### Chart 5 — regularisation tuning curve

In [ ]:
gp.plot_validation_curve(gp.C_GRID, cv_means, cv_stds);

## 7 · AFTER training — train the final model

In [ ]:
from sklearn.metrics import classification_report
import time

model = gp.make_model(best_C)
t0 = time.time()
model.fit(X_tr, y_tr)
print(f"trained in {time.time()-t0:.2f}s")

y_pred = model.predict(X_te)
y_prob = model.predict_proba(X_te)[:, 1]
trained = gp.evaluate(model, X_te, y_te)

print("\n" + "="*60)
print("CLASSIFICATION REPORT (held-out test set)")
print("="*60)
print(classification_report(y_te, y_pred, target_names=gp.CLASS_NAMES, digits=4))

### Chart 6 — BEFORE vs AFTER

The single most important chart in your demo.

In [ ]:
results = {**baselines, "TRAINED\nmodel": trained}
gp.plot_baselines(results);

### Chart 7 — confusion matrix

In [ ]:
gp.plot_confusion(y_te, y_pred);

### Chart 8 — ROC and precision-recall curves

In [ ]:
fig, auc, ap = gp.plot_roc_pr(y_te, y_prob)
print(f"ROC-AUC = {auc:.4f}   Average Precision = {ap:.4f}")

### Chart 9 — cross-validation

Five folds with a standard deviation is much stronger evidence than one split.
**Report these numbers as your headline result**, not the single test score.

In [ ]:
cv = gp.cv_scores(gp.make_model(best_C), X, y, folds=5)
print("fold scores:", np.round(cv, 4))
print(f"mean macro F1 = {cv.mean():.4f} +/- {cv.std():.4f}")
gp.plot_cv(cv);

### Chart 10 — learning curve

**This answers "are 1000 images enough?"**

- Curves converged and flat → more data will not help; change the features/model.
- Validation still climbing at the right edge → more data *will* help.
- Big gap between train and validation → still overfitting.

In [ ]:
fig, sizes, tr_sc, te_sc = gp.plot_learning_curve(gp.make_model(best_C), X, y)
print(f"train F1 at max size: {tr_sc[-1]:.4f}")
print(f"valid F1 at max size: {te_sc[-1]:.4f}")
print(f"gap: {tr_sc[-1]-te_sc[-1]:.4f}   (under ~0.10 is healthy)")

### Chart 11 — subgroup accuracy (bias check)

Face classifiers routinely perform worse on darker skin tones. Measuring this
and reporting it honestly earns more credit than a higher headline number.

In [ ]:
gp.plot_groups(y_te, y_pred, g_te);

### Chart 12 — confidence gating

Connects the model to your actual product. Above the threshold, pre-select the
style; below it, open the manual picker with nothing chosen.

In [ ]:
gp.plot_confidence(y_te, y_pred, y_prob);

## 8 · Export for the browser

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import os, json

onx = convert_sklearn(model,
                      initial_types=[("input", FloatTensorType([None, X.shape[1]]))],
                      target_opset=13,
                      options={id(model): {"zipmap": False}})
open("guardian_style.onnx", "wb").write(onx.SerializeToString())
print(f"guardian_style.onnx  ({os.path.getsize('guardian_style.onnx')/1024:.1f} KB)")

metrics = {
    "dataset": {"files_found": audit["files_found"], "labelled": audit["labelled"],
                "cleaning": reasons, "balance_before": {str(k): v for k, v in balance_before.items()},
                "balance_after": {str(k): v for k, v in balance_after.items()}},
    "best_C": best_C,
    "baselines": {k.replace("\n", " "): v for k, v in baselines.items()},
    "trained": trained,
    "cross_validation": {"folds": cv.tolist(), "mean": float(cv.mean()), "std": float(cv.std())},
    "roc_auc": float(auc), "average_precision": float(ap),
}
json.dump(metrics, open("metrics.json", "w"), indent=2)
print("metrics.json written")

## 9 · Download everything

Grabs all 12 charts plus the model and metrics as one zip — no screenshotting
needed, though the inline charts above screenshot fine too.

In [ ]:
!zip -qr safequest_results.zip *.png guardian_style.onnx metrics.json
from google.colab import files
files.download("safequest_results.zip")